# ECHR framing analysis — parental alienation / contact disputes

**Corpus:** the ECHR JSON only (`data/echr_parental_alienation.json`). RIS / Swiss / German data are **not** touched here.

**Question:** across the whole ECHR corpus, how are parental-alienation / contact-dispute situations *framed*? E.g.

- as a **state failure** to enforce contact (Article 8 positive obligations),
- as a **psychological** phenomenon (alienation, manipulation, loyalty conflict),
- as **cultural / religious** estrangement (the Kilic-type framing),
- through the lens of the **child's rights** (best interests, child's wishes),
- or something the lexicons miss.

**Two parts, kept separate:**

- **Part A — lexicon-based framing prevalence** (primary, defensible): a *distributional* measure. Editable framing lexicons → per-case hit counts → a dominant framing per case → corpus prevalence. This distribution **is** the answer, with the standing caveat that *lexicon choice drives the result*.
- **Part B — unsupervised clustering** (secondary, exploratory): embed judgments' reasoning, KMeans for a few k, inspect top terms — purely to reveal framings the lexicons don't capture. **Not** presented as definitive framings.

**Document types are never pooled.** The corpus mixes full Court **judgments**, **communicated cases** (Registry templates: *SUBJECT MATTER OF THE CASE* / *QUESTIONS TO THE PARTIES*), and admissibility/strike-out **decisions**. Their boilerplate differs and would create false clusters, so every result below is reported per document type.

**Constraints:** CPU-only (2015 Intel Mac); plain-text + counts output; defensive coding (`.get`, never assume keys).

> **Run order for confirmation:** the notebook prints the *judgment-vs-communicated split*, the *section-parsing success rate*, and *lexicon hits on 3 example cases* **before** the full Part A run, so the lexicons can be confirmed against the real data first.

In [1]:
# --- imports (stdlib + numpy/pandas/matplotlib for Part A; ML libs imported lazily in Part B) ---
from pathlib import Path
from collections import Counter, defaultdict
import re, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # headless save only, no interactive windows
import matplotlib.pyplot as plt

# --- project paths (root = parent of src/) ---
ROOT = Path.cwd()
if ROOT.name == "src":
    ROOT = ROOT.parent
DATA, FIGURES, REPORTS = ROOT / "data", ROOT / "figures", ROOT / "reports"
ECHR_PATH = DATA / "echr_parental_alienation.json"
print("ROOT      :", ROOT)
print("ECHR file :", ECHR_PATH, "| exists:", ECHR_PATH.exists())

ROOT      : /Users/maksimsmirnov/Desktop/thesis
ECHR file : /Users/maksimsmirnov/Desktop/thesis/data/echr_parental_alienation.json | exists: True


## Configuration — editable framing lexicons

These dicts **drive Part A**. Edit them and re-run; nothing downstream is hard-coded to specific terms.

Matching rules (so you know what each entry does):

- multi-word entries match the words in sequence with flexible whitespace (`facilitate contact` → `facilitate\s+contact`);
- a `*` is a stem wildcard *anywhere* in a token (`reunit*` → `reunite`, `reunited`, `reunification`; `estrange* from culture` → `estranged from culture`, …);
- matching is case-insensitive and counts **every** occurrence in the section (not just presence).

The slash entry from the brief (`influence of the mother/father`) is split into two explicit entries below.

In [2]:
# === EDITABLE LEXICONS — refine these, then re-run from here down ===
LEXICONS = {
    "state_failure": [
        "enforcement", "non-enforcement", "positive obligation", "failure to assist",
        "bailiff", "authorities failed to", "facilitate contact", "reunite", "take measures",
    ],
    "psychological": [
        "alienation", "alienate", "manipulate", "turn the child against", "loyalty conflict",
        "refuses contact", "emotional bond", "psychological pressure",
        "influence of the mother", "influence of the father",
    ],
    "cultural_religious": [
        "cultural", "religious", "identity", "estrange* from culture", "language",
        "ethnic", "foster family",
    ],
    "childs_rights": [
        "best interests of the child", "child's wishes", "child's views",
        "child's right to maintain contact",
    ],
}

# --- dominant-framing tuning ---
MIXED_RATIO = 0.80   # if 2nd framing's normalised score >= MIXED_RATIO * top, label the case "mixed"

# --- Part B tuning (used much later) ---
K_VALUES      = (3, 4, 5, 6)                              # KMeans k values to scan
E5_MODEL      = "intfloat/multilingual-e5-base"           # CPU-only, already cached locally
EMB_CACHE     = DATA / "echr_framing_emb_cache.npz"       # judgment-assessment embeddings
DO_DEDUP      = True                                      # strip near-duplicate boilerplate sentences before embedding
DEDUP_DOC_FRAC = 0.30                                     # drop a sentence if it appears verbatim in >= this fraction of judgments
for f, terms in LEXICONS.items():
    print(f"{f:18s} {len(terms)} terms")

state_failure      9 terms
psychological      10 terms
cultural_religious 7 terms
childs_rights      4 terms


## Step 1 — Load & classify document types

Three document types are detected from `doctype` (cross-checked against `doctypebranch` / `conclusion`):

- `HEJUD` → **judgment** (full Court judgment, reasoning under *THE LAW*),
- `HECOM` → **communicated** case (Registry template),
- `HEDEC` → **decision** (admissibility / strike-out).

These are reported and analysed **separately** throughout.

In [3]:
with open(ECHR_PATH, encoding="utf-8") as fh:
    RECORDS = json.load(fh)

print("N records:", len(RECORDS))
print("\nFields on first record:")
print("  " + ", ".join(sorted((RECORDS[0] if RECORDS else {}).keys())))

DOCTYPE_TO_BUCKET = {"HEJUD": "judgment", "HECOM": "communicated", "HEDEC": "decision"}
def classify_bucket(rec):
    return DOCTYPE_TO_BUCKET.get((rec.get("doctype") or "").upper(), "other")

for rec in RECORDS:
    rec["_bucket"] = classify_bucket(rec)

bucket_counts = Counter(r["_bucket"] for r in RECORDS)
print("\n=== DOCUMENT-TYPE SPLIT (never pooled) ===")
for b in ["judgment", "communicated", "decision", "other"]:
    if bucket_counts.get(b):
        print(f"  {bucket_counts[b]:4d}  {b}")

print("\ndoctype x doctypebranch (cross-check):")
ct = Counter((r.get("doctype"), r.get("doctypebranch")) for r in RECORDS)
for (dt, br), n in ct.most_common():
    print(f"  {n:4d}  {dt or '(none)':8s} | {br or '(none)'}")

print("\nTop 'conclusion' values:")
for v, n in Counter((r.get("conclusion") or "(empty)") for r in RECORDS).most_common(8):
    print(f"  {n:4d}  {v[:80]}")

print("\nLanguages:", dict(Counter((r.get("languageisocode") or "?") for r in RECORDS)))

N records: 719

Fields on first record:
  applicability, appno, article, conclusion, docname, doctype, doctypebranch, ecli, externalsources, extractedappno, full_text, importance, issue, itemid, judgementdate, jurisdiction, lang, languageisocode, matched_keywords, nonviolation, originatingbody, publishedby, rank, referencedate, representedby, respondent, retrieved_date, rulesofcourt, scl, separateopinion, source, source_url, stable_id, text_length, violation

=== DOCUMENT-TYPE SPLIT (never pooled) ===
   361  judgment
   188  communicated
   170  decision

doctype x doctypebranch (cross-check):
   268  HEJUD    | CHAMBER
   188  HECOM    | COMMUNICATEDCASES
   138  HEDEC    | ADMISSIBILITYCOM
    76  HEJUD    | COMMITTEE
    32  HEDEC    | ADMISSIBILITY
    17  HEJUD    | GRANDCHAMBER

Top 'conclusion' values:
   188  Communicated
   147  Inadmissible
    80  Violation of Article 8 - Right to respect for private and family life (Article 8
    27  No violation of Article 8 - Right to re

## Step 2 — Parse the reasoning / framing section

We search only the part of each document where the framing lives, not the whole text:

- **judgments / decisions:** everything from the **`THE LAW`** heading onward (the legal reasoning, incl. *The Court's assessment*); if absent, fall back to *The Court's assessment*; else the full text (flagged).
- **communicated cases:** the **`SUBJECT MATTER OF THE CASE`** section (up to *QUESTIONS TO THE PARTIES*), i.e. the Registry's factual framing. The templated *QUESTIONS* block is deliberately excluded so its boilerplate doesn't dominate. If no anchor parses, the full text is used (flagged).

The raw text has **no line breaks** and uses non-breaking spaces and curly apostrophes, so we normalise first. The cell prints the **parse success rate per document type** and how many cases fell back to full text.

In [4]:
_NORMALISE = [("\xa0", " "), ("’", "'"), ("‘", "'"), ("“", '"'), ("”", '"')]
def normalise_text(t):
    t = t or ""
    for a, b in _NORMALISE:
        t = t.replace(a, b)
    return t

_RE_THE_LAW    = re.compile(r"THE LAW")                          # uppercase heading only
_RE_ASSESSMENT = re.compile(r"the court's assessment", re.I)

def reasoning_section(rec):
    """Return (section_text, parse_status). parse_status flags fallbacks."""
    text = normalise_text(rec.get("full_text"))
    if not text:
        return "", "empty"
    bucket = rec.get("_bucket")
    if bucket == "communicated":
        low = text.lower()
        i = low.find("subject matter of the case")
        j = low.find("questions to the parties")
        if i >= 0:
            end = j if j > i else len(text)
            return text[i:end].strip(), "subject_matter"
        if j >= 0:
            return text[:j].strip(), "pre_questions"
        return text, "fallback_full"
    # judgments / decisions / other
    m = _RE_THE_LAW.search(text)
    if m:
        return text[m.start():].strip(), "the_law"
    m = _RE_ASSESSMENT.search(text)
    if m:
        return text[m.start():].strip(), "court_assessment"
    return text, "fallback_full"

for rec in RECORDS:
    sec, status = reasoning_section(rec)
    rec["_section"] = sec
    rec["_parse_status"] = status

print("=== SECTION-PARSING SUCCESS RATE (per document type) ===")
FALLBACK = {"fallback_full", "empty"}
for b in ["judgment", "communicated", "decision", "other"]:
    recs = [r for r in RECORDS if r["_bucket"] == b]
    if not recs:
        continue
    parsed = sum(1 for r in recs if r["_parse_status"] not in FALLBACK)
    print(f"\n{b}  (n={len(recs)}): parsed {parsed}/{len(recs)}  "
          f"({100*parsed/len(recs):.0f}%) | fell back to full text: {len(recs)-parsed}")
    for st, n in Counter(r["_parse_status"] for r in recs).most_common():
        flag = "  <-- FALLBACK" if st in FALLBACK else ""
        print(f"    {n:4d}  {st}{flag}")

seclen = [len(r["_section"]) for r in RECORDS if r["_section"]]
print(f"\nSection length chars: min={min(seclen)} median={int(np.median(seclen))} max={max(seclen)}")

=== SECTION-PARSING SUCCESS RATE (per document type) ===

judgment  (n=361): parsed 361/361  (100%) | fell back to full text: 0
     325  the_law
      36  court_assessment

communicated  (n=188): parsed 164/188  (87%) | fell back to full text: 24
      95  subject_matter
      69  pre_questions
      24  fallback_full  <-- FALLBACK

decision  (n=170): parsed 170/170  (100%) | fell back to full text: 0
      97  the_law
      73  court_assessment

Section length chars: min=30 median=14149 max=760893


## Part A — Lexicon-based framing prevalence (primary)

For each case we count lexicon hits in the parsed section, normalise each framing's count by the **number of terms in that lexicon** (so a larger lexicon doesn't win automatically), and assign a **dominant framing**:

- `none` — no lexicon hit at all;
- `mixed` — the runner-up's normalised score is ≥ `MIXED_RATIO` × the top score;
- otherwise the top framing.

Counting happens within one shared section per case, so section length cancels out when comparing framings *within* a case; the lexicon-size normalisation is the meaningful adjustment.

In [5]:
def build_lexicon_patterns(lexicons):
    """term -> compiled regex; '*' is a stem wildcard anywhere, words join with flexible whitespace."""
    compiled = {}
    for frame, terms in lexicons.items():
        pats = []
        for term in terms:
            toks = [re.escape(tok).replace(r"\*", r"\w*") for tok in term.lower().split()]
            body = r"\s+".join(toks)
            pats.append((term, re.compile(r"(?<!\w)" + body, re.IGNORECASE)))
        compiled[frame] = pats
    return compiled

PATTERNS = build_lexicon_patterns(LEXICONS)

def frame_counts(text, patterns=PATTERNS, per_term=False):
    """Raw occurrence counts per framing; optionally which terms fired."""
    totals, fired = {}, {}
    for frame, pats in patterns.items():
        c = 0
        hits = {}
        for term, pat in pats:
            k = len(pat.findall(text))
            if k:
                hits[term] = k
            c += k
        totals[frame] = c
        fired[frame] = hits
    return (totals, fired) if per_term else totals

def dominant_frame(counts, lexicons=LEXICONS, mixed_ratio=MIXED_RATIO):
    """Return (label, normalised_scores). Normalise raw counts by lexicon size."""
    norm = {f: counts.get(f, 0) / max(1, len(lexicons[f])) for f in lexicons}
    ranked = sorted(norm.items(), key=lambda kv: -kv[1])
    top_f, top_s = ranked[0]
    if top_s == 0:
        return "none", norm
    if len(ranked) > 1 and ranked[1][1] >= mixed_ratio * top_s and ranked[1][1] > 0:
        return "mixed", norm
    return top_f, norm

print("Lexicon patterns built for framings:", ", ".join(PATTERNS))

Lexicon patterns built for framings: state_failure, psychological, cultural_religious, childs_rights


### Confirmation checkpoint — lexicon hits on 3 example cases

Before the full run, inspect the matcher on one **judgment**, one **communicated** case and one **decision**: which terms fire, the per-framing raw counts, and the assigned dominant framing. **Confirm the lexicons match the real text here before trusting the distribution below.**

In [6]:
def show_case(rec):
    counts, fired = frame_counts(rec["_section"], per_term=True)
    label, norm = dominant_frame(counts)
    print("=" * 78)
    print(f"{rec.get('docname','(no name)')}  [{rec.get('appno','')}]")
    print(f"  bucket={rec['_bucket']} | parse={rec['_parse_status']} | section chars={len(rec['_section'])}")
    print(f"  raw counts : " + "  ".join(f"{f}={counts[f]}" for f in LEXICONS))
    print(f"  normalised : " + "  ".join(f"{f}={norm[f]:.2f}" for f in LEXICONS))
    print(f"  DOMINANT   : {label}")
    for f in LEXICONS:
        if fired[f]:
            print(f"    {f}: " + ", ".join(f"{t}x{n}" for t, n in fired[f].items()))

# pick the first record in each bucket that actually parsed to a real section
examples = []
for b in ["judgment", "communicated", "decision"]:
    for r in RECORDS:
        if r["_bucket"] == b and r["_parse_status"] not in FALLBACK and len(r["_section"]) > 400:
            examples.append(r)
            break
for r in examples:
    show_case(r)

CASE OF G.B. v. LITHUANIA  [36137/13]
  bucket=judgment | parse=the_law | section chars=41406
  raw counts : state_failure=38  psychological=9  cultural_religious=0  childs_rights=8
  normalised : state_failure=4.22  psychological=0.90  cultural_religious=0.00  childs_rights=2.00
  DOMINANT   : state_failure
    state_failure: enforcementx7, non-enforcementx2, positive obligationx5, bailiffx18, authorities failed tox3, take measuresx3
    psychological: alienationx7, alienatex2
    childs_rights: best interests of the childx7, child's wishesx1
KICHEVA v. BULGARIA  [77760/14]
  bucket=communicated | parse=pre_questions | section chars=15883
  raw counts : state_failure=41  psychological=5  cultural_religious=0  childs_rights=0
  normalised : state_failure=4.56  psychological=0.50  cultural_religious=0.00  childs_rights=0.00
  DOMINANT   : state_failure
    state_failure: enforcementx11, bailiffx29, reunitex1
    psychological: alienationx3, manipulatex1, psychological pressurex1
Y AND O

### Full Part A run — framing prevalence across the corpus

Reported **per document type**:

1. **Dominant-framing distribution** — the headline prevalence answer.
2. **Coverage** — for each framing, the share of cases mentioning it at least once (independent of which framing "wins").
3. **Example case names** per dominant framing, for sanity-checking.

> **Caveat (always read with the numbers):** these proportions are a function of the lexicons above. They describe how often *these terms* surface, which is a defensible proxy for framing prevalence — not a ground-truth label of each case.

In [7]:
rows = []
for rec in RECORDS:
    counts = frame_counts(rec["_section"])
    label, norm = dominant_frame(counts)
    row = {"docname": rec.get("docname", ""), "appno": rec.get("appno", ""),
           "bucket": rec["_bucket"], "parse": rec["_parse_status"],
           "dominant": label, "respondent": rec.get("respondent", "")}
    row.update({f"n_{f}": counts[f] for f in LEXICONS})
    rows.append(row)
df = pd.DataFrame(rows)

FRAME_ORDER = list(LEXICONS) + ["mixed", "none"]
print("=== 1. DOMINANT-FRAMING DISTRIBUTION (per document type) ===")
for b in ["judgment", "communicated", "decision"]:
    sub = df[df.bucket == b]
    if sub.empty:
        continue
    print(f"\n{b}  (n={len(sub)})")
    vc = sub.dominant.value_counts()
    for f in FRAME_ORDER:
        if f in vc.index:
            print(f"  {vc[f]:4d}  ({100*vc[f]/len(sub):4.0f}%)  {f}")

print("\n=== 2. COVERAGE — % of cases with >=1 hit for each framing ===")
for b in ["judgment", "communicated", "decision"]:
    sub = df[df.bucket == b]
    if sub.empty:
        continue
    print(f"\n{b}  (n={len(sub)})")
    for f in LEXICONS:
        cov = (sub[f"n_{f}"] > 0).mean()
        print(f"  {100*cov:4.0f}%  {f}")

print("\n=== 3. EXAMPLE CASES per dominant framing (judgments) ===")
jud = df[df.bucket == "judgment"]
for f in FRAME_ORDER:
    names = jud[jud.dominant == f].docname.head(5).tolist()
    if names:
        print(f"\n  [{f}]")
        for nm in names:
            print(f"    - {nm}")

=== 1. DOMINANT-FRAMING DISTRIBUTION (per document type) ===

judgment  (n=361)
   163  (  45%)  state_failure
     3  (   1%)  psychological
    52  (  14%)  cultural_religious
   104  (  29%)  childs_rights
    30  (   8%)  mixed
     9  (   2%)  none

communicated  (n=188)
    65  (  35%)  state_failure
     3  (   2%)  psychological
    24  (  13%)  cultural_religious
    28  (  15%)  childs_rights
    11  (   6%)  mixed
    57  (  30%)  none

decision  (n=170)
    48  (  28%)  state_failure
     2  (   1%)  psychological
    24  (  14%)  cultural_religious
    45  (  26%)  childs_rights
     5  (   3%)  mixed
    46  (  27%)  none

=== 2. COVERAGE — % of cases with >=1 hit for each framing ===

judgment  (n=361)
    85%  state_failure
    26%  psychological
    45%  cultural_religious
    73%  childs_rights

communicated  (n=188)
    45%  state_failure
    12%  psychological
    22%  cultural_religious
    28%  childs_rights

decision  (n=170)
    51%  state_failure
     8%  psych

In [8]:
# Optional single bar of dominant-framing prevalence among JUDGMENTS (saved, not shown).
jud = df[df.bucket == "judgment"]
vc = jud.dominant.value_counts().reindex([f for f in FRAME_ORDER if f in set(jud.dominant)]).fillna(0)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(vc)), vc.values, color="#4C72B0")
ax.set_xticks(range(len(vc)))
ax.set_xticklabels(vc.index, rotation=30, ha="right")
ax.set_ylabel("judgments")
ax.set_title(f"ECHR framing prevalence — dominant framing among judgments (n={len(jud)})")
for i, v in enumerate(vc.values):
    ax.text(i, v, f"{int(v)}", ha="center", va="bottom", fontsize=9)
fig.tight_layout()
out = FIGURES / "echr_framing_prevalence.png"
fig.savefig(out, dpi=130)
plt.close(fig)
print("saved:", out)

saved: /Users/maksimsmirnov/Desktop/thesis/figures/echr_framing_prevalence.png


## Part B — Unsupervised clustering (secondary, **exploratory only**)

> **This is a cross-check, not a result.** Its only purpose is to surface a framing the lexicons might miss. Clusters are **not** definitive framings.
>
> Expect caveats to bite: **boilerplate recitals** ("The Court reiterates that…") may still dominate clusters even after dedup; **small / uneven cluster sizes** are normal; the e5 model truncates long sections to ~512 tokens, so only the start of each reasoning section is embedded.

Scope: **judgments only** (`HEJUD`). Communicated cases and decisions are excluded so Registry/admissibility boilerplate can't form clusters. We embed each judgment's reasoning section (preferring the tighter *The Court's assessment* subsection when present), optionally strip near-duplicate boilerplate sentences, run KMeans for several k, and print per-cluster top distinctive terms (class-based TF-IDF) plus example cases.

In [9]:
# Build the judgment set for clustering: prefer the tighter "The Court's assessment" subsection.
def assessment_for_clustering(rec):
    text = normalise_text(rec.get("full_text"))
    m = _RE_ASSESSMENT.search(text)
    if m:
        return text[m.start():].strip()
    return rec.get("_section", "")

clu = [r for r in RECORDS if r["_bucket"] == "judgment" and r.get("_section")]
clu_texts = [assessment_for_clustering(r) for r in clu]
clu_names = [r.get("docname", "") for r in clu]
clu_ids   = [r.get("stable_id") or r.get("itemid") or str(i) for i, r in enumerate(clu)]
print(f"judgments for clustering: {len(clu)}")
print("assessment-section chars: "
      f"min={min(len(t) for t in clu_texts)} "
      f"median={int(np.median([len(t) for t in clu_texts]))} "
      f"max={max(len(t) for t in clu_texts)}")

judgments for clustering: 361
assessment-section chars: min=3760 median=24294 max=760787


In [10]:
# Optional near-duplicate boilerplate removal: drop sentences appearing verbatim in many judgments.
_SENT = re.compile(r"(?<=[.;])\s+")
def split_sentences(t):
    return [s.strip() for s in _SENT.split(t) if len(s.strip()) > 25]

if DO_DEDUP:
    doc_freq = Counter()
    for t in clu_texts:
        for s in set(split_sentences(t)):
            doc_freq[s] += 1
    cutoff = max(2, int(DEDUP_DOC_FRAC * len(clu_texts)))
    boiler = {s for s, n in doc_freq.items() if n >= cutoff}
    cleaned = []
    for t in clu_texts:
        keep = [s for s in split_sentences(t) if s not in boiler]
        cleaned.append(" ".join(keep) if keep else t)
    before = int(np.median([len(t) for t in clu_texts]))
    after  = int(np.median([len(t) for t in cleaned]))
    print(f"dedup: removed {len(boiler)} boilerplate sentences "
          f"(>= {cutoff}/{len(clu_texts)} docs); median chars {before} -> {after}")
    embed_texts = cleaned
else:
    embed_texts = clu_texts
    print("dedup disabled")

dedup: removed 0 boilerplate sentences (>= 108/361 docs); median chars 24294 -> 23875


In [11]:
# Embed with multilingual-e5-base (CPU). Cached to EMB_CACHE keyed by stable id.
def load_cache(path, ids):
    if not path.exists():
        return None
    d = np.load(path, allow_pickle=True)
    if list(d["ids"]) == list(ids):
        print("loaded cached embeddings:", path)
        return d["emb"]
    print("cache id-mismatch -> recompute")
    return None

emb = load_cache(EMB_CACHE, clu_ids)
if emb is None:
    from sentence_transformers import SentenceTransformer
    print("loading model (CPU):", E5_MODEL)
    model = SentenceTransformer(E5_MODEL, device="cpu")
    # e5 expects a "passage: " prefix for documents
    emb = model.encode(["passage: " + t for t in embed_texts],
                       batch_size=8, normalize_embeddings=True, show_progress_bar=False)
    emb = np.asarray(emb, dtype=np.float32)
    np.savez(EMB_CACHE, ids=np.array(clu_ids, dtype=object), emb=emb)
    print("saved embeddings:", EMB_CACHE)
print("embeddings shape:", emb.shape)

loaded cached embeddings: /Users/maksimsmirnov/Desktop/thesis/data/echr_framing_emb_cache.npz
embeddings shape: (361, 768)


In [12]:
# KMeans scan + class-based TF-IDF top terms + example cases. Exploratory cross-check only.
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import silhouette_score

LEGAL_STOP = {"court","article","applicant","applicants","government","convention",
              "paragraph","paragraphs","para","case","cases","mr","mrs","ms","also",
              "would","may","must","said","thus","upon","shall","whether","under"}
stop = list(TfidfVectorizer(stop_words="english").get_stop_words() | LEGAL_STOP)

for k in K_VALUES:
    km = KMeans(n_clusters=k, random_state=0, n_init=10).fit(emb)
    labels = km.labels_
    sil = silhouette_score(emb, labels) if len(set(labels)) > 1 else float("nan")
    print("\n" + "#" * 78)
    print(f"# KMeans k={k}  | sizes={dict(Counter(labels))}  | silhouette={sil:.3f}")
    # class-based TF-IDF: one pseudo-document per cluster
    cluster_docs = [" ".join(embed_texts[i] for i in range(len(embed_texts)) if labels[i] == c)
                    for c in range(k)]
    vec = TfidfVectorizer(stop_words=stop, max_features=4000, ngram_range=(1, 2), min_df=2)
    X = vec.fit_transform(cluster_docs)
    vocab = np.array(vec.get_feature_names_out())
    for c in range(k):
        row = X[c].toarray().ravel()
        top = vocab[row.argsort()[::-1][:12]]
        # example cases = nearest to centroid (cosine == dot, embeddings are normalised)
        members = [i for i in range(len(labels)) if labels[i] == c]
        sims = emb[members] @ km.cluster_centers_[c]
        ex = [clu_names[members[j]] for j in np.argsort(sims)[::-1][:3]]
        print(f"\n  cluster {c}  (n={len(members)})")
        print(f"    top terms: {', '.join(top)}")
        for nm in ex:
            print(f"    e.g. {nm}")


##############################################################################
# KMeans k=3  | sizes={2: 125, 0: 105, 1: 131}  | silhouette=0.039

  cluster 0  (n=105)
    top terms: child, domestic, family, children, authorities, cited, respect, courts, above_, proceedings, interests, contact
    e.g. CASE OF GARAI v. HUNGARY
    e.g. CASE OF E.S. v. ROMANIA AND BULGARIA
    e.g. CASE OF SEVERE v. AUSTRIA

  cluster 1  (n=131)
    top terms: cited, domestic, rights, state, law, authorities, above_, proceedings, child, present, right, cited above_
    e.g. CASE OF ŠIRVINSKAS v. LITHUANIA
    e.g. CASE OF JÍROVÁ AND OTHERS v. THE CZECH REPUBLIC
    e.g. CASE OF KUKAVICA v. BULGARIA

  cluster 2  (n=125)
    top terms: child, family, domestic, authorities, contact, children, respect, interests, rights, proceedings, cited, above_
    e.g. CASE OF K.E. AND A.K. v. NORWAY
    e.g. CASE OF L.D. v. POLAND
    e.g. CASE OF M.L. v. NORWAY

######################################################

### Reading Part B

- Treat the clusters as **prompts for inspection**, not categories. If a cluster's top terms point at a theme absent from the Part A lexicons (e.g. abduction/return, domestic violence, expert evidence), that is the signal worth acting on — add or refine a lexicon in the config cell and re-run Part A.
- Cluster top terms heavy with procedural language ("reiterates", "margin of appreciation", "just satisfaction") indicate **residual boilerplate**, not a framing — raise `DEDUP_DOC_FRAC` or ignore that cluster.
- `silhouette` near 0 means the judgments don't separate cleanly in embedding space — expected for a thematically narrow corpus, and a further reason not to over-read the clusters.

The defensible deliverable of this notebook is **Part A's prevalence table**; Part B only guards against lexicon blind spots.